# Daylan+2017 Fermi-LAT point-source analysis

This notebook runs PCAT on a synthetic North Galactic Pole field with the paper's 300-source mock population, three energy bins, two conversion classes, and $-1.8$ flux slope. The quick run uses a smaller pixel grid and fewer samples. Its figures follow selected mock-run figure types from the paper, but the exposure and diffuse template are synthetic, so they do not reproduce Fermi observations or the paper's numerical posterior.

In [1]:
from pcat.demo import load_example_namespace

example = load_example_namespace(
    'daylan+2017_fermi_point_sources/daylan+2017_fermi_point_sources.py'
)
run = example['run_reproduction'](quick=True)
posterior = example['read_posterior'](run)

In [2]:
import numpy as np

# Use one retained catalog and the injected catalog from this same PCAT run.
energy_bin = 1  # [1-3 GeV]
sample_index = len(posterior.listpostdictelem) // 2
sample_catalog = posterior.listpostdictelem[sample_index][0]
true_catalog = {
    name: np.asarray(posterior.refr.dictelem[0][name])[0]
    for name in ('xpos', 'ypos', 'flux')
}
print(f'Retained sample {sample_index} of {len(posterior.listpostdictelem)}')
print(f'Injected sources: {len(true_catalog["flux"])}')
print(f'Sampled sources: {len(sample_catalog["flux"])}')

Retained sample 500 of 1000
Injected sources: 40
Sampled sources: 20


In [3]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from tdpy.verbosity import print

plt.rcParams.update({
    'font.size': 10,
    'text.usetex': False,
    'axes.grid': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
})

visuals_path = Path(example['OUTPUT_ROOT']) / 'visuals'
visuals_path.mkdir(parents=True, exist_ok=True)
number_side = int(np.sqrt(posterior.cntpdata.shape[1]))
half_width_deg = example['FIELD_HALF_WIDTH_DEG']
radian_to_degree = np.rad2deg(1.0)  # [deg rad^-1]


def save_and_show(figure, filename):
    path = visuals_path / filename
    print(f'Writing to {path}...')
    figure.savefig(path, dpi=300, bbox_inches='tight')
    display(figure)
    plt.close(figure)


# Figure 5-style comparison of simulated data, one fair catalog draw, and residuals.
data_map = posterior.cntpdata[energy_bin].sum(axis=-1).reshape(number_side, number_side)
model_map = posterior.listpostcntpmodl[sample_index, energy_bin].sum(axis=-1).reshape(
    number_side, number_side
)
residual_map = (data_map - model_map) / np.sqrt(np.maximum(model_map, 1.0))
stretch_scale = max(float(np.median(data_map[data_map > 0])), 1.0)
extent = (-half_width_deg, half_width_deg, -half_width_deg, half_width_deg)
figure, axes = plt.subplots(1, 3, figsize=(10.5, 3.5), constrained_layout=True)
for axis, image, title in zip(
    axes[:2], (data_map, model_map), ('Simulated counts, 1-3 GeV', 'One posterior draw')
):
    artist = axis.imshow(
        np.arcsinh(image / stretch_scale), origin='lower', extent=extent, cmap='magma',
        vmin=0.0, vmax=np.arcsinh(max(float(data_map.max()), float(model_map.max())) / stretch_scale),
    )
    axis.scatter(
        np.asarray(true_catalog['xpos']) * radian_to_degree,
        np.asarray(true_catalog['ypos']) * radian_to_degree,
        marker='+', s=18, linewidths=0.5, color='lime', label='Injected sources',
    )
    axis.scatter(
        np.asarray(sample_catalog['xpos']) * radian_to_degree,
        np.asarray(sample_catalog['ypos']) * radian_to_degree,
        marker='x', s=12, linewidths=0.5, color='deepskyblue', label='Sampled sources',
    )
    axis.set_title(title)
    axis.set_xlabel('Angular offset [deg]')
    axis.set_ylabel('Angular offset [deg]')
    axis.legend(loc='upper right', fontsize=7, framealpha=1.0)
    figure.colorbar(artist, ax=axis, label=r'arcsinh(counts / %.2g)' % stretch_scale)
artist = axes[2].imshow(
    residual_map, origin='lower', extent=extent, cmap='coolwarm',
    vmin=-5.0, vmax=5.0,
)
axes[2].set_title('Poisson residual')
axes[2].set_xlabel('Angular offset [deg]')
axes[2].set_ylabel('Angular offset [deg]')
figure.colorbar(artist, ax=axes[2], label='Residual [model standard deviations]')
save_and_show(figure, 'fermi_mock_figure5_style.png')

# Figure 7-style comparison of the injected and posterior flux distributions.
list_flux = [np.asarray(catalog[0]['flux']) for catalog in posterior.listpostdictelem]
positive_flux = np.concatenate([values[values > 0.0] for values in list_flux if values.size])
true_flux = np.asarray(true_catalog['flux'])
flux_edges = np.geomspace(min(positive_flux.min(), true_flux.min()), max(positive_flux.max(), true_flux.max()), 18)
list_flux_counts = np.stack([
    np.histogram(values, bins=flux_edges)[0] for values in list_flux
])
flux_quantiles = np.percentile(list_flux_counts, [16, 50, 84], axis=0)
flux_bin_width = np.diff(np.log10(flux_edges))
flux_bin_center = np.sqrt(flux_edges[:-1] * flux_edges[1:])
figure, axis = plt.subplots(figsize=(6.4, 4.2), constrained_layout=True)
axis.fill_between(
    flux_bin_center, flux_quantiles[0] / flux_bin_width,
    flux_quantiles[2] / flux_bin_width, step='mid', color='C0', alpha=0.25,
    label='Posterior 68% interval',
)
axis.step(
    flux_bin_center, flux_quantiles[1] / flux_bin_width, where='mid', color='C0',
    label='Posterior median',
)
axis.step(
    flux_bin_center, np.histogram(true_flux, bins=flux_edges)[0] / flux_bin_width,
    where='mid', color='C2', label='Injected catalog',
)
axis.set_xscale('log')
axis.set_yscale('log')
axis.set_xlabel('Point-source flux [cm$^{-2}$ s$^{-1}$ GeV$^{-1}$]')
axis.set_ylabel('Sources per dex')
axis.legend(loc='best', framealpha=1.0)
save_and_show(figure, 'fermi_mock_figure7_style.png')

# Figure 11-style posterior for the number of point sources.
source_count = np.asarray(posterior.listpostnumbelem).ravel().astype(int)
count_values, count_frequency = np.unique(source_count, return_counts=True)
figure, axis = plt.subplots(figsize=(5.2, 3.6), constrained_layout=True)
axis.bar(count_values, count_frequency / count_frequency.sum(), color='C0', label='Posterior')
axis.axvline(len(true_flux), color='C2', linestyle='--', label='Injected count')
axis.set_xlabel('Number of point sources')
axis.set_ylabel('Posterior probability')
axis.legend(loc='best', framealpha=1.0)
save_and_show(figure, 'fermi_mock_figure11_style.png')

log_likelihood = np.asarray(posterior.listpostlliktotl).ravel()
figure, axis = plt.subplots(figsize=(7.0, 3.2), constrained_layout=True)
axis.plot(np.arange(log_likelihood.size), log_likelihood, color='C0', linewidth=0.8)
axis.set_xlabel('Retained posterior sample')
axis.set_ylabel('Log likelihood')
save_and_show(figure, 'fermi_mock_log_likelihood.png')

print('Figures show a synthetic mock analysis; they do not reproduce the paper data.')

<Figure size 2100x700 with 6 Axes>

<Figure size 1280x840 with 1 Axes>

<Figure size 1040x720 with 1 Axes>

<Figure size 1400x640 with 1 Axes>